In [1]:
import os

import kagglehub
import pandas as pd
import polars as pl

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(-1)

from kagglehub import KaggleDatasetAdapter

# Downloading dataset

In [2]:
def download_data_to_pandas():
    path = kagglehub.dataset_download("usdot/flight-delays")
    flights_path = "flights.csv"
    airlines_path = "airlines.csv"
    airports_path = "airports.csv"
    print("Reading dataset: ", path)
    flights_pd = pd.read_csv(
        os.path.join(path, flights_path),
        encoding="ISO-8859-1",
        low_memory=False,
        dtype={
            'YEAR': 'category',
            'DAY': 'category',
            'MONTH': 'category',
            'DAY_OF_WEEK': 'category',
            'AIRLINE': 'category',
            'FLIGHT_NUMBER': 'category',
            'TAIL_NUMBER': 'category',
            'ORIGIN_AIRPORT': 'category',
            'DESTINATION_AIRPORT': 'category',

            'SCHEDULED_DEPARTURE': 'str',
            'DEPARTURE_TIME': 'str',

            'WHEELS_OFF': 'str',
            'WHEELS_ON': 'str',

            'SCHEDULED_ARRIVAL': 'str',
            'ARRIVAL_TIME': 'str',

            'CANCELLATION_REASON': 'category',
        }
    )
    airlines_pd = pd.read_csv(
        os.path.join(path, airlines_path),
        encoding="ISO-8859-1",
        low_memory=False,
        dtype={
            'IATA_CODE': 'category',
            'AIRLINE': 'category',
        }
    )
    airports_pd = pd.read_csv(
        os.path.join(path, airports_path),
        encoding="ISO-8859-1",
        low_memory=False,
        dtype={
            'IATA_CODE': 'category',
            'AIRPORT': 'category',
            'CITY': 'category',
            'STATE': 'category',
            'COUNTRY': 'category',

        }
    )
    return flights_pd, airlines_pd, airports_pd

def download_data_to_polars():
    path = kagglehub.dataset_download("usdot/flight-delays")
    flights_path = "flights.csv"
    airlines_path = "airlines.csv"
    airports_path = "airports.csv"
    print("Reading dataset: ", path)

    flights_pl = pl.read_csv(
        os.path.join(path, flights_path),
        low_memory=False,
        encoding="ISO-8859-1",
        schema_overrides = {
            'YEAR': pl.Categorical,
            'DAY': pl.Categorical,
            'MONTH': pl.Categorical,
            'DAY_OF_WEEK': pl.Categorical,
            'AIRLINE': pl.Categorical,
            'FLIGHT_NUMBER': pl.Categorical,
            'TAIL_NUMBER': pl.Categorical,
            'ORIGIN_AIRPORT': pl.Categorical,
            'DESTINATION_AIRPORT': pl.Categorical,

            'SCHEDULED_DEPARTURE': pl.String,
            'DEPARTURE_TIME': pl.String,

            'WHEELS_OFF': pl.String,
            'WHEELS_ON': pl.String,

            'SCHEDULED_ARRIVAL': pl.String,
            'ARRIVAL_TIME': pl.String,

            'CANCELLATION_REASON': pl.Categorical,
        }
    )
        #TODO - add creating dataframes for other datasets


def download_bigger():
    file_path = "2015.csv"
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018",
        file_path,
        pandas_kwargs={"encoding": "ISO-8859-1",
                       "low_memory": False,
                       "compression": 'zip'
                       }

    )
    return df

In [3]:
flights, airlines, airports = download_data_to_pandas()

Reading dataset:  C:\Users\kb412\.cache\kagglehub\datasets\usdot\flight-delays\versions\1


# Airports

In [ ]:
airports.head()

In [ ]:
print("No. nulls in airports:\n", airports.isna().sum())

In [ ]:
airports.loc[airports.isna().any(axis=1)]

There are 3 empty coordinates in the airports data. This kind of nulls can be inserted manually, after checking the coordinates.

In [ ]:
airports.loc[airports['IATA_CODE'] == "ECP", ["LATITUDE", 'LONGITUDE']] = [30.35824, -85.79560]
airports.loc[airports['IATA_CODE'] == "PBG", ["LATITUDE", 'LONGITUDE']] = [44.65094, -73.46814]
airports.loc[airports['IATA_CODE'] == "UST", ["LATITUDE", 'LONGITUDE']] = [29.95925, -81.33973]
print("No. nulls in airports:\n", airports.isna().any().sum())

# Airlines

In [8]:
airlines.head()

,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.
3,F9,Frontier Airlines Inc.
4,B6,JetBlue Airways


In [9]:
print("No. nulls in airlines:\n", airlines.isna().sum())

No. nulls in airlines:
 IATA_CODE    0
AIRLINE      0
dtype: int64


Luckily, there are no empty rows in the airlines dataset

# Flights

In [10]:
import polars as pl

flights = pl.from_pandas(flights).lazy()
airlines = pl.from_pandas(airlines).lazy()
airports = pl.from_pandas(airports).lazy()

## Initial insight into the data

In [11]:
flights.head(10).collect()

YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
cat,cat,cat,cat,cat,cat,cat,cat,cat,str,str,f64,f64,str,f64,f64,f64,i64,str,f64,str,str,f64,i64,i64,cat,f64,f64,f64,f64,f64
"""2015""","""1""","""1""","""4""","""AS""","""98""","""N407AS""","""ANC""","""SEA""","""0005""","""2354""",-11.0,21.0,"""0015""",205.0,194.0,169.0,1448,"""0404""",4.0,"""0430""","""0408""",-22.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""AA""","""2336""","""N3KUAA""","""LAX""","""PBI""","""0010""","""0002""",-8.0,12.0,"""0014""",280.0,279.0,263.0,2330,"""0737""",4.0,"""0750""","""0741""",-9.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""US""","""840""","""N171US""","""SFO""","""CLT""","""0020""","""0018""",-2.0,16.0,"""0034""",286.0,293.0,266.0,2296,"""0800""",11.0,"""0806""","""0811""",5.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""AA""","""258""","""N3HYAA""","""LAX""","""MIA""","""0020""","""0015""",-5.0,15.0,"""0030""",285.0,281.0,258.0,2342,"""0748""",8.0,"""0805""","""0756""",-9.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""AS""","""135""","""N527AS""","""SEA""","""ANC""","""0025""","""0024""",-1.0,11.0,"""0035""",235.0,215.0,199.0,1448,"""0254""",5.0,"""0320""","""0259""",-21.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""DL""","""806""","""N3730B""","""SFO""","""MSP""","""0025""","""0020""",-5.0,18.0,"""0038""",217.0,230.0,206.0,1589,"""0604""",6.0,"""0602""","""0610""",8.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""NK""","""612""","""N635NK""","""LAS""","""MSP""","""0025""","""0019""",-6.0,11.0,"""0030""",181.0,170.0,154.0,1299,"""0504""",5.0,"""0526""","""0509""",-17.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""US""","""2013""","""N584UW""","""LAX""","""CLT""","""0030""","""0044""",14.0,13.0,"""0057""",273.0,249.0,228.0,2125,"""0745""",8.0,"""0803""","""0753""",-10.0,0,0,null,null,null,null,null,null
"""2015""","""1""","""1""","""4""","""AA""","""1112""","""N3LAAA""","""SFO""","""DFW""","""0030""","""0019""",-11.0,17.0,"""0036""",195.0,193.0,173.0,1464,"""0529""",3.0,"""0545""","""0532""",-13.0,0,0,null,null,null,null,null,null


In [12]:
flights.collect_schema().names()

['YEAR',
 'MONTH',
 'DAY',
 'DAY_OF_WEEK',
 'AIRLINE',
 'FLIGHT_NUMBER',
 'TAIL_NUMBER',
 'ORIGIN_AIRPORT',
 'DESTINATION_AIRPORT',
 'SCHEDULED_DEPARTURE',
 'DEPARTURE_TIME',
 'DEPARTURE_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'SCHEDULED_TIME',
 'ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'WHEELS_ON',
 'TAXI_IN',
 'SCHEDULED_ARRIVAL',
 'ARRIVAL_TIME',
 'ARRIVAL_DELAY',
 'DIVERTED',
 'CANCELLED',
 'CANCELLATION_REASON',
 'AIR_SYSTEM_DELAY',
 'SECURITY_DELAY',
 'AIRLINE_DELAY',
 'LATE_AIRCRAFT_DELAY',
 'WEATHER_DELAY']

In [13]:
(flights.describe())

statistic,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,str,f64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64
"""count""","""5819079""","""5819079""","""5819079""","""5819079""","""5819079""","""5819079""","""5804358""","""5819079""","""5819079""","""5819079""","""5732926""",5.732926e6,5.730032e6,"""5730032""",5.819073e6,5.714008e6,5.714008e6,5.819079e6,"""5726566""",5.726566e6,"""5819079""","""5726566""",5.714008e6,5.819079e6,5.819079e6,"""89884""",1.063439e6,1.063439e6,1.063439e6,1.063439e6,1.063439e6
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""14721""","""0""","""0""","""0""","""86153""",86153.0,89047.0,"""89047""",6.0,105071.0,105071.0,0.0,"""92513""",92513.0,"""0""","""92513""",105071.0,0.0,0.0,"""5729195""",4.75564e6,4.75564e6,4.75564e6,4.75564e6,4.75564e6
"""mean""",null,null,null,null,null,null,null,null,null,null,null,9.370158,16.071662,null,141.685892,137.006189,113.511628,822.356495,null,7.434971,null,null,4.407057,0.00261,0.015446,null,13.480568,0.076154,18.969547,23.472838,2.91529
"""std""",null,null,null,null,null,null,null,null,null,null,null,37.080942,8.895574,null,75.210582,74.211072,72.230822,607.784287,null,5.638548,null,null,39.271297,0.05102,0.12332,null,28.003679,2.14346,48.161642,43.197018,20.433336
"""min""",null,null,null,null,null,null,null,null,null,"""0001""","""0001""",-82.0,1.0,"""0001""",18.0,14.0,7.0,21.0,"""0001""",1.0,"""0001""","""0001""",-87.0,0.0,0.0,null,0.0,0.0,0.0,0.0,0.0
"""25%""",null,null,null,null,null,null,null,null,null,null,null,-5.0,11.0,null,85.0,82.0,60.0,373.0,null,4.0,null,null,-13.0,0.0,0.0,null,0.0,0.0,0.0,0.0,0.0
"""50%""",null,null,null,null,null,null,null,null,null,null,null,-2.0,14.0,null,123.0,118.0,94.0,647.0,null,6.0,null,null,-5.0,0.0,0.0,null,2.0,0.0,2.0,3.0,0.0
"""75%""",null,null,null,null,null,null,null,null,null,null,null,7.0,19.0,null,173.0,168.0,144.0,1062.0,null,9.0,null,null,8.0,0.0,0.0,null,18.0,0.0,19.0,29.0,0.0
"""max""",null,null,null,null,null,null,null,null,null,"""2359""","""2400""",1988.0,225.0,"""2400""",718.0,766.0,690.0,4983.0,"""2400""",248.0,"""2400""","""2400""",1971.0,1.0,1.0,null,1134.0,573.0,1971.0,1331.0,1211.0


In [14]:
flights.collect_schema().dtypes()

[Categorical,
 Categorical,
 Categorical,
 Categorical,
 Categorical,
 Categorical,
 Categorical,
 Categorical,
 Categorical,
 String,
 String,
 Float64,
 Float64,
 String,
 Float64,
 Float64,
 Float64,
 Int64,
 String,
 Float64,
 String,
 String,
 Float64,
 Int64,
 Int64,
 Categorical,
 Float64,
 Float64,
 Float64,
 Float64,
 Float64]

## Changing time columns to appropriate data type

In [15]:
time_columns = [
    'SCHEDULED_DEPARTURE',
    'DEPARTURE_TIME',
    'WHEELS_OFF',
    'WHEELS_ON',
    'SCHEDULED_ARRIVAL',
    'ARRIVAL_TIME',
]
schema = flights.collect_schema()

for col in time_columns:
    if schema[col] == pl.String:
        flights = flights.with_columns(
            pl.col(col)
            .str.zfill(4)
            .str.replace("2400", "0000")
            .str.to_time("%H%M", strict=False).alias(col)
        )


## Dealing with nulls

In [16]:
print("No. nulls in airlines:\n",
      flights
      .null_count()
      .unpivot(variable_name='tmp', value_name='null_count')
      .filter(pl.col('null_count') > 0)
      .collect())

No. nulls in airlines:
 shape: (18, 2)
┌─────────────────────┬────────────┐
│ tmp                 ┆ null_count │
│ ---                 ┆ ---        │
│ str                 ┆ u32        │
╞═════════════════════╪════════════╡
│ TAIL_NUMBER         ┆ 14721      │
│ DEPARTURE_TIME      ┆ 86153      │
│ DEPARTURE_DELAY     ┆ 86153      │
│ TAXI_OUT            ┆ 89047      │
│ WHEELS_OFF          ┆ 89047      │
│ SCHEDULED_TIME      ┆ 6          │
│ ELAPSED_TIME        ┆ 105071     │
│ AIR_TIME            ┆ 105071     │
│ WHEELS_ON           ┆ 92513      │
│ TAXI_IN             ┆ 92513      │
│ ARRIVAL_TIME        ┆ 92513      │
│ ARRIVAL_DELAY       ┆ 105071     │
│ CANCELLATION_REASON ┆ 5729195    │
│ AIR_SYSTEM_DELAY    ┆ 4755640    │
│ SECURITY_DELAY      ┆ 4755640    │
│ AIRLINE_DELAY       ┆ 4755640    │
│ LATE_AIRCRAFT_DELAY ┆ 4755640    │
│ WEATHER_DELAY       ┆ 4755640    │
└─────────────────────┴────────────┘


### Tail number

We can try to start dealing with null on Tail Numbers.

There is  possibility that some Airline doesn't use many different Tail numbers.

In [17]:
(flights.group_by('AIRLINE')
 .agg(pl.col('TAIL_NUMBER')
      .n_unique()
      .alias('unique_tail_numbers'))
 .sort('unique_tail_numbers')
 .collect())

AIRLINE,unique_tail_numbers
cat,u32
"""HA""",51
"""VX""",57
"""F9""",63
"""NK""",79
"""AS""",147
"""MQ""",204
"""B6""",215
"""US""",352
"""OO""",384


Imputing with mode might be impossible for all. We can try to impute this type of data using knn. There is a possibility where there are e.g., many flights that are basically the same but on the other day.

In [18]:
print(flights
      .filter(
    ~pl.col('TAIL_NUMBER').is_null(),
    pl.col('ORIGIN_AIRPORT') == 'ANC',
    pl.col('DESTINATION_AIRPORT') == 'DEN',
    pl.col('AIRLINE') == 'UA',
    pl.col('DAY_OF_WEEK') == '1',
    pl.col('FLIGHT_NUMBER') == '1104'
)
      .select(pl.col('TAIL_NUMBER')
              .value_counts())
      .unnest('TAIL_NUMBER')
      .sort('count', descending=True)
      .collect())


shape: (44, 2)
┌─────────────┬───────┐
│ TAIL_NUMBER ┆ count │
│ ---         ┆ ---   │
│ cat         ┆ u32   │
╞═════════════╪═══════╡
│ N76519      ┆ 2     │
│ N73270      ┆ 2     │
│ N73299      ┆ 1     │
│ N66828      ┆ 1     │
│ N39297      ┆ 1     │
│ N76254      ┆ 1     │
│ N27205      ┆ 1     │
│ N73256      ┆ 1     │
│ N14250      ┆ 1     │
│ N73275      ┆ 1     │
│ N17245      ┆ 1     │
│ N36247      ┆ 1     │
│ N38417      ┆ 1     │
│ N16234      ┆ 1     │
│ N66825      ┆ 1     │
│ N37267      ┆ 1     │
│ N36207      ┆ 1     │
│ N78285      ┆ 1     │
│ N26210      ┆ 1     │
│ N37427      ┆ 1     │
│ N33292      ┆ 1     │
│ N76269      ┆ 1     │
│ N76528      ┆ 1     │
│ N14231      ┆ 1     │
│ N37273      ┆ 1     │
│ N14214      ┆ 1     │
│ N76522      ┆ 1     │
│ N38424      ┆ 1     │
│ N12225      ┆ 1     │
│ N17233      ┆ 1     │
│ N37437      ┆ 1     │
│ N38473      ┆ 1     │
│ N78509      ┆ 1     │
│ N68811      ┆ 1     │
│ N76508      ┆ 1     │
│ N18223      ┆ 1     │
│

It looks like that imputation might be challenging, there aren't any repetitive tail numbers that can be used for imputation.
Tail number might not be important for predicting, maybe only for visualization, so we can drop this column in a further part or drop rows when visualizing.

### Cancellation reason

The next column for nulls analysis might be Cancellation reason with 5729195 nulls. Reason why those nulls occur is that those flights weren't canceled, so there is no reason, we can create new class "None" which can represent those nulls.
The number of not null values can also tell us many useful information for further imputations.

In [19]:
flights = flights.with_columns(
    pl.col('CANCELLATION_REASON').fill_null("None")
)

### Delays

There is very similar situation with delays, if there is a Null it means there was none delay, so we can impute this blanks with 0.

In [20]:
flights = flights.with_columns(
    pl.col('AIR_SYSTEM_DELAY').fill_null(0),
    pl.col('SECURITY_DELAY').fill_null(0),
    pl.col('AIRLINE_DELAY').fill_null(0),
    pl.col('LATE_AIRCRAFT_DELAY').fill_null(0),
    pl.col('WEATHER_DELAY').fill_null(0)
)

### Diverted and canceled analysis

In [21]:
(flights.select(
    ((pl.col('DIVERTED') == 1) | (pl.col('CANCELLED') == 1)).sum().alias('diverted_or_cancelled'),
    pl.col('DIVERTED').sum().alias('diverted'),
    pl.col('CANCELLED').sum().alias('cancelled')
).collect())

diverted_or_cancelled,diverted,cancelled
u32,i64,i64
105071,15187,89884


There are some columns that are empty because of flight being canceled or diverted, specifically:
- │ ELAPSED_TIME        ┆ 105071     │
- │ AIR_TIME            ┆ 105071     │
- │ ARRIVAL_DELAY       ┆ 105071     │

This is because  these columns aren't inserted automatically by system and there is no sense to calculate delay or time in the air if the plane flew to another airport or didn't left origin one. We won't impute them with anything because it won't have any sense, if we want to train model for e.g., delays we should drop rows diverted or canceled flights.

#### Diverted

In [22]:
print("No. nulls in airlines:\n",
      flights
      .filter((pl.col('DIVERTED') == 1))
      .null_count()
      .unpivot(variable_name='tmp', value_name='null_count')
      .filter(pl.col('null_count') > 0)
      .collect())

No. nulls in airlines:
 shape: (7, 2)
┌────────────────┬────────────┐
│ tmp            ┆ null_count │
│ ---            ┆ ---        │
│ str            ┆ u32        │
╞════════════════╪════════════╡
│ SCHEDULED_TIME ┆ 1          │
│ ELAPSED_TIME   ┆ 15187      │
│ AIR_TIME       ┆ 15187      │
│ WHEELS_ON      ┆ 2629       │
│ TAXI_IN        ┆ 2629       │
│ ARRIVAL_TIME   ┆ 2629       │
│ ARRIVAL_DELAY  ┆ 15187      │
└────────────────┴────────────┘


There are 15187 diverted flights, and there is the same number of blanks in all columns apart from WHEELS_ON, TAXI_IN and ARRIVAL_TIME, as it was said before in other columns there is no sens to insert them.

The number of nulls in these 3 columns is showing us that most of the flights that were diverted finally reached the destination, but 2629 of them didn't and that's why columns aren't filled.

#### Cancelled


In [23]:
print("No. nulls in airlines:\n",
      flights
      .filter((pl.col('CANCELLED') == 1))
      .null_count()
      .unpivot(variable_name='tmp', value_name='null_count')
      .filter(pl.col('null_count') > 0)
      .collect())

No. nulls in airlines:
 shape: (12, 2)
┌─────────────────┬────────────┐
│ tmp             ┆ null_count │
│ ---             ┆ ---        │
│ str             ┆ u32        │
╞═════════════════╪════════════╡
│ TAIL_NUMBER     ┆ 14721      │
│ DEPARTURE_TIME  ┆ 86153      │
│ DEPARTURE_DELAY ┆ 86153      │
│ TAXI_OUT        ┆ 89047      │
│ WHEELS_OFF      ┆ 89047      │
│ SCHEDULED_TIME  ┆ 5          │
│ ELAPSED_TIME    ┆ 89884      │
│ AIR_TIME        ┆ 89884      │
│ WHEELS_ON       ┆ 89884      │
│ TAXI_IN         ┆ 89884      │
│ ARRIVAL_TIME    ┆ 89884      │
│ ARRIVAL_DELAY   ┆ 89884      │
└─────────────────┴────────────┘


As we can see there are many nulls at canceled flights.

Now we can analyze all of them except TAIL_NUMBER and SCHEDULED_TIME.

In [24]:
q1 = flights.filter(
    (pl.col('CANCELLED') == 1),
    pl.col('DEPARTURE_TIME').is_null()
).head(1).collect()
q2 = flights.filter(
    (pl.col('CANCELLED') == 1),
    pl.col('TAXI_OUT').is_null(),
    ~pl.col('DEPARTURE_TIME').is_null()
).head(1).collect()
q3 = flights.filter(
    (pl.col('CANCELLED') == 1),
    pl.col('ELAPSED_TIME').is_null(),
    ~pl.col('WHEELS_OFF').is_null()
).head(1).collect()
pl.concat([q1, q2, q3])

YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
cat,cat,cat,cat,cat,cat,cat,cat,cat,time,time,f64,f64,time,f64,f64,f64,i64,time,f64,time,time,f64,i64,i64,cat,f64,f64,f64,f64,f64
"""2015""","""1""","""1""","""4""","""AS""","""136""","""N431AS""","""ANC""","""SEA""",01:35:00,null,null,null,null,205.0,null,null,1448,null,null,06:00:00,null,null,0,1,"""A""",0.0,0.0,0.0,0.0,0.0
"""2015""","""1""","""1""","""4""","""MQ""","""2758""","""N939MQ""","""ROW""","""DFW""",11:10:00,12:29:00,79.0,null,null,85.0,null,null,435,null,null,13:35:00,null,null,0,1,"""B""",0.0,0.0,0.0,0.0,0.0
"""2015""","""1""","""1""","""4""","""EV""","""4503""","""N14514""","""IAH""","""BRO""",11:34:00,11:29:00,-5.0,20.0,11:49:00,76.0,null,null,308,null,null,12:50:00,null,null,0,1,"""B""",0.0,0.0,0.0,0.0,0.0


As we can see, there are three types of nulls.

- Departure time is null, the plane didn't even leave the terminal, so other columns are also empty.
- Departure time is not null, but wheels off are empty, it means that the plane left the terminal but didn't take off.
- This time the plane took off, so the previous stats are populated, but it didn't land or destination airport nor was diverted,

So any of these null aren't unwanted and it's normal behavior.

There are also nulls in SCHEDULED_TIME which should be investigated. Looking at the amount of them they probably shouldn't exist. It is also occurring in one of the diverted flight.

In [25]:
flights.filter(pl.col('SCHEDULED_TIME').is_null()).collect()

YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
cat,cat,cat,cat,cat,cat,cat,cat,cat,time,time,f64,f64,time,f64,f64,f64,i64,time,f64,time,time,f64,i64,i64,cat,f64,f64,f64,f64,f64
"""2015""","""2""","""1""","""7""","""NK""","""416""","""N627NK""","""FLL""","""IAG""",22:15:00,22:06:00,-9.0,21.0,22:27:00,null,null,null,1176,null,null,01:07:00,null,null,0,1,"""B""",0.0,0.0,0.0,0.0,0.0
"""2015""","""2""","""10""","""2""","""NK""","""416""","""N632NK""","""FLL""","""IAG""",22:15:00,22:06:00,-9.0,16.0,22:22:00,null,null,null,1176,null,null,01:07:00,null,null,0,1,"""B""",0.0,0.0,0.0,0.0,0.0
"""2015""","""4""","""20""","""1""","""NK""","""174""","""N630NK""","""FLL""","""LGA""",16:02:00,18:57:00,175.0,24.0,19:21:00,null,null,null,1076,null,null,19:00:00,null,null,1,0,"""None""",0.0,0.0,0.0,0.0,0.0
"""2015""","""4""","""26""","""7""","""NK""","""720""","""N614NK""","""DEN""","""DFW""",20:59:00,20:57:00,-2.0,13.0,21:10:00,null,null,null,641,null,null,23:50:00,null,null,0,1,"""B""",0.0,0.0,0.0,0.0,0.0
"""2015""","""5""","""9""","""6""","""NK""","""324""","""N620NK""","""MCO""","""ACY""",21:30:00,21:52:00,22.0,18.0,22:10:00,null,null,null,852,null,null,23:40:00,null,null,0,1,"""B""",0.0,0.0,0.0,0.0,0.0
"""2015""","""5""","""10""","""7""","""NK""","""368""","""N534NK""","""ORD""","""BWI""",08:25:00,12:14:00,229.0,18.0,12:32:00,null,null,null,622,null,null,11:18:00,null,null,0,1,"""C""",0.0,0.0,0.0,0.0,0.0


As we can see, everything that should be inserted is there, besides scheduled time.

It can be imputed using mean for specific airports pair. Values will be calculated using only data from certain origin and destination and these means will be inserted in place of nulls.

In [26]:
airports_w_nulls = (flights.filter(pl.col('SCHEDULED_TIME').is_null())
                    .select('ORIGIN_AIRPORT', 'DESTINATION_AIRPORT')
                    .unique()
                    .collect()
                    )
medians = (
    flights
    .join(
        airports_w_nulls.lazy(),
        on=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'],
        how='inner'
    )
    .filter(
        (~pl.col('SCHEDULED_TIME').is_null())
    )
    .group_by('ORIGIN_AIRPORT', 'DESTINATION_AIRPORT')
    .agg(pl.col("SCHEDULED_TIME")
         .mean()
         .round(1)
         .alias('times median'))
    .sort('times median', descending=False)
    .collect()
)

flights_imputed = (flights
    .join(
        medians.lazy(),
        on=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'],
        how='left'
    )
    .with_columns(
    pl.col('SCHEDULED_TIME').fill_null(pl.col('times median'))
    )
    .drop('times median')
)

### Normal flights analysis

We should also look at the normal flights, if there are any nulls.

In [27]:
print("No. nulls in airlines:\n",
      flights
      .filter((pl.col('CANCELLED') == 0),pl.col('DIVERTED')==0)
      .null_count()
      .unpivot(variable_name='tmp', value_name='null_count')
      .filter(pl.col('null_count') > 0)
      .collect())

No. nulls in airlines:
 shape: (0, 2)
┌─────┬────────────┐
│ tmp ┆ null_count │
│ --- ┆ ---        │
│ str ┆ u32        │
╞═════╪════════════╡
└─────┴────────────┘


There are no nulls in these flights.

# Saving preprocessed dataset to parquet files

In [29]:
flights.sink_parquet('data/flights.parquet')
airlines.sink_parquet('data/airlines.parquet')
airports.sink_parquet('data/airports.parquet')

rsconnect deploy shiny "C:\Users\kb412\PycharmProjects\flight-prediction-analysis" --name bubakrojek --title flights-analysis
